# Normalize Morita et al. lipid names to LipidMaps format

**Goal:** Match lipid names from Morita et al. to LipidMaps entries (LM_ID + SMILES) using a cascade of name-format conversions.

**Steps:**
1. Get LipidMaps library via `pypath_v2` (local, no API required)
2. Load Morita data and normalize names (preserving `_` as chain separator)
3. Match normalized names to LipidMaps using a cascade conversion strategy
4. Calculate and print match rate
5. Save results to `data/processed/lipidmaps_Morita.csv`

**`_` vs `/` (LIPID MAPS Shorthand Nomenclature):**
- `_` = fatty acid composition known, sn-position **unknown** (Morita raw data uses `_` exclusively)
- `/` = sn-position **confirmed** — not introduced in intermediate names for glycerophospholipids
- **Sphingolipid exception:** LipidMaps stores Cer/SM/HexCer only with `/` in d-notation (0 `_` entries).
  The `_`→`/` conversion in `to_cer_d_notation` is intentional — see code comment for biological justification.

**Naming format differences:**

| Level | Morita (raw) | Intermediate | LipidMaps (DB key) | Sep rule |
|---|---|---|---|---|
| Molecular species | `TG(12:0)(14:1)(18:2)` | `TG 12:0_14:1_18:2` | `TG(12:0_14:1_18:2)` | `_` preserved |
| Molecular species | `PC 16:0_18:1` | `PC 16:0_18:1` | `PC(16:0_18:1)` | `_` preserved |
| Ceramide | `Cer d18:1_16:0` | `Cer 18:1;2_16:0` | `Cer(d18:1/16:0)` | `_`→`/` (sphingolipid exception) |
| SM (species) | `SM 36:1` | `SM 36:1` | `SM 36:1;O2` | species-level |


In [60]:
import re
import pandas as pd
from collections import Counter

## Step 1 — Get LipidMaps library via pypath_v2

Build a dictionary mapping **lipid name → list of {lm_id, smiles}**.
Both `ABBREVIATION` and `SYNONYMS` fields are indexed as keys so that
different naming styles from LipidMaps are all searchable.

Only entries with both a valid `LM_ID` and a `SMILES` string are included.

In [61]:
import os, pickle

CACHE_PATH = '../data/processed/cache_lipidmaps_db.pkl'

if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, 'rb') as f:
        lipid_db = pickle.load(f)
    print(f'Loaded from cache : {len(lipid_db):,} keys  ({CACHE_PATH})')
    print('Delete the cache file to force a fresh download.')
else:
    from pypath.inputs_v2.lipidmaps import resource

    # name -> list of {'lm_id': ..., 'smiles': ...}
    lipid_db = {}

    for raw in resource.lipids.raw():
        lm_id  = raw.get('LM_ID')
        smiles = raw.get('SMILES')
        if not lm_id or not smiles:
            continue

        names = set()
        if raw.get('ABBREVIATION'):
            names.add(raw['ABBREVIATION'])
        if raw.get('SYNONYMS'):
            names.update(raw['SYNONYMS'].split(';'))

        entry = {'lm_id': lm_id, 'smiles': smiles}
        for name in names:
            name = name.strip()
            if name:
                lipid_db.setdefault(name, []).append(entry)

    with open(CACHE_PATH, 'wb') as f:
        pickle.dump(lipid_db, f)

    print(f'Built and cached   : {len(lipid_db):,} keys')
    print(f'Cache saved to     : {CACHE_PATH}')

print(f'Example keys: {list(lipid_db.keys())[100:105]}')

Loaded from cache : 36,711 keys  (../data/processed/cache_lipidmaps_db.pkl)
Delete the cache file to force a fresh download.
Example keys: ['n-Eicosanoic acid', 'C20:0', 'Eicosanoate', 'Arachic acid', 'Henicosanoic acid']


## Step 1.5 — Load Morita data and drop dummy rows

Rows where `ExactMass` is empty are not measured lipids but dummy/placeholder rows.  
Remove them before any name matching.

In [62]:
df_raw = pd.read_excel('../data/raw/lipid_Morita_et_al.xlsx', sheet_name='Lipid_all')

n_before = len(df_raw)
df_raw = df_raw[df_raw['ExactMass'].notna()].reset_index(drop=True)
n_after  = len(df_raw)

print(f'Rows before filtering : {n_before}')
print(f'Rows dropped (dummy)  : {n_before - n_after}')
print(f'Rows after filtering  : {n_after}')
df_raw[['CompoundName', 'MolecularFormula', 'ExactMass']].head(5)

Rows before filtering : 1306
Rows dropped (dummy)  : 17
Rows after filtering  : 1289


,CompoundName,MolecularFormula,ExactMass
0,Cer d18:1_14:0,C32H63N1O3,509.480795
1,Cer d18:1_16:0,C34H67N1O3,537.512095
2,Cer d18:1_16:1,C34H65N1O3,535.496445
3,Cer d18:1_18:0,C36H71N1O3,565.543395
4,Cer d18:1_18:1,C36H69N1O3,563.527745


## Step 2 — Load Morita data and normalize lipid names

Morita et al. uses a proprietary naming convention.
We apply two regex rules to convert it to an intermediate format for LipidMaps cascade matching.

**`_` vs `/` (LIPID MAPS Shorthand Nomenclature):**
- `_` = fatty acid composition known, sn-position **unknown** (Morita raw data uses `_` exclusively)
- `/` = sn-position **confirmed** — must NOT be introduced unless present in the raw data

| Rule | Input | Output | Note |
|---|---|---|---|
| Parenthesis chains | `TG(12:0)(14:1)(18:2)` | `TG 12:0_14:1_18:2` | `_` preserved (sn-position unknown) |
| Ceramide `d` prefix | `Cer d18:1_16:0` | `Cer 18:1;2_16:0` | `d`→`;2` is equivalent notation; `_` preserved |

Rule 3 (underscore → slash) has been **removed**: it fabricated sn-position information absent from the raw data.

In [63]:
def normalize_lipid_name(name):
    """
    Convert Morita et al. proprietary lipid names to an intermediate format
    for LipidMaps cascade matching.

    '_' vs '/':
      '_' = fatty acid composition known, sn-position UNKNOWN  (Morita raw data)
      '/' = sn-position CONFIRMED — must NOT be introduced unless present in raw data

    Rules applied in order:
    1. Parenthesis chains -> underscore-separated (molecular species level)
       e.g. TG(12:0)(14:1)(18:2) -> TG 12:0_14:1_18:2
    2. Ceramide 'd' prefix -> ';2' hydroxyl notation  (equivalent, not fabrication)
       e.g. Cer d18:1_16:0 -> Cer 18:1;2_16:0
       '_' is preserved as chain separator (NOT converted to '/')
    """
    # Rule 1: parenthesis-enclosed chains — use '_', not '/'
    m = re.match(r'^([A-Za-z][A-Za-z0-9]*)((?:\([^)]+\))+)$', name)
    if m:
        cls    = m.group(1)
        chains = re.findall(r'\(([^)]+)\)', m.group(2))
        name   = cls + ' ' + '_'.join(chains)

    # Rule 2: Ceramide 'd' prefix — preserve '_' as chain separator
    name = re.sub(r'((?:Hex)?Cer) d(\d+:\d+)_(\S+)', r'\1 \2;2_\3', name)

    return name


lipid_names      = df_raw['CompoundName'].dropna().unique().tolist()
lipid_names_norm = [normalize_lipid_name(n) for n in lipid_names]

print(f'Total unique Morita lipid names: {len(lipid_names)}')
print()
print('Normalization examples:')
for orig, norm in zip(lipid_names[:6], lipid_names_norm[:6]):
    print(f'  {orig:35s} -> {norm}')

# Verify: no '_'-containing original produces a '/'-containing normalized name
violations_norm = [
    (o, n) for o, n in zip(lipid_names, lipid_names_norm)
    if '_' in o and '/' in n
]
if violations_norm:
    print(f'\nWARNING: {len(violations_norm)} normalization violations (_->/)!')
    for o, n in violations_norm[:5]:
        print(f'  {o} -> {n}')
else:
    print('\nOK: normalize_lipid_name preserves _ in all names.')

Total unique Morita lipid names: 1289

Normalization examples:
  Cer d18:1_14:0                      -> Cer 18:1;2_14:0
  Cer d18:1_16:0                      -> Cer 18:1;2_16:0
  Cer d18:1_16:1                      -> Cer 18:1;2_16:1
  Cer d18:1_18:0                      -> Cer 18:1;2_18:0
  Cer d18:1_18:1                      -> Cer 18:1;2_18:1
  Cer d18:1_18:2                      -> Cer 18:1;2_18:2

OK: normalize_lipid_name preserves _ in all names.


## Step 3 — Match normalized names to LipidMaps (cascade strategy)

LipidMaps stores lipid names in several formats.
We try each conversion in priority order and return the first hit.

**`_` vs `/` separator rule:**
- For most lipid classes, `_`-input generates `_`-keyed lookups (and vice versa).
  LipidMaps contains both `PC(16:0_18:1)` and `PC(16:0/18:1)` as distinct entries.
- **Exception — sphingolipids (Cer, SM, HexCer, ...):**
  LipidMaps registers these with `/` only (verified: 0 entries with `_`).
  Biologically, the sphingoid base sn-position is fixed by the amide bond, so
  LipidMaps treats all ceramide molecular subspecies as structural subspecies.
  `to_cer_d_notation` therefore converts `_` input to `/` DB key intentionally.

| Priority | Strategy | Input → LipidMaps DB key | Sep rule |
|---|---|---|---|
| 1 | Exact match | `PC 16:0_18:1` → `PC 16:0_18:1` | preserves `_` |
| 2 | Parentheses + slash | `PC 16:0/18:1` → `PC(16:0/18:1)` | `/`-input only |
| 3 | Parentheses + underscore | `TG 12:0_14:1_18:2` → `TG(12:0_14:1_18:2)` | `_`-input only |
| 4 | Ceramide d-notation | `Cer 18:1;2_16:0` → `Cer(d18:1/16:0)` | `_`→`/` (sphingolipid exception) |
| 5 | Species-level (total C:DB) | `PC 16:0_18:1` → `PC 36:1`, `Cer 18:1;2_16:0` → `Cer 34:1;O2` | no chain sep |

Since Morita data uses `_` exclusively, strategy 2 (paren + slash) is effectively inactive for this dataset.


In [64]:
SPHINGOLIPID_CLASSES = {'Cer', 'SM', 'HexCer', 'Hex2Cer', 'LacCer', 'GlcCer'}


def _split_chains(name):
    """Return (class_prefix, [chain_strings], sep) for a normalized lipid name.

    sep = '/' — sn-position specified (Structural subspecies, LIPID MAPS '/')
    sep = '_' — chain composition only (Molecular subspecies, LIPID MAPS '_')
    sep = None — single-chain lipid (no separator present)

    Preserving sep is critical: converting '/' to '_' or vice versa changes the
    structural level and causes incorrect database matches.
    """
    m = re.match(r'^([A-Za-z]+(?:\s[A-Za-z]+)?)\s+(.+)$', name)
    if not m:
        return None, [], None
    cls       = m.group(1)
    chain_str = m.group(2)

    if '/' in chain_str:
        sep    = '/'
        chains = [c.strip() for c in chain_str.split('/')]
    elif re.search(r'\d_\d', chain_str):
        sep    = '_'
        chains = [c.strip() for c in re.split(r'(?<=\d)_(?=\d)', chain_str)]
    else:
        sep    = None
        chains = [chain_str.strip()]

    return cls, chains, sep


def to_parentheses_slash(name):
    """Structural subspecies: 'PC 16:0/18:1' -> 'PC(16:0/18:1)' (sn-position known)."""
    cls, chains, sep = _split_chains(name)
    if cls is None or len(chains) < 2 or sep != '/':
        return None
    return f"{cls}({'/'.join(chains)})"


def to_parentheses_underscore(name):
    """Molecular subspecies: 'TG 12:0_14:1_18:2' -> 'TG(12:0_14:1_18:2)' (sn-position unknown)."""
    cls, chains, sep = _split_chains(name)
    if cls is None or len(chains) < 2 or sep != '_':
        return None
    return f"{cls}({'_'.join(chains)})"


def to_cer_d_notation(name):
    """
    Convert sphingolipid ';2' notation to LipidMaps 'd'-prefix with slash separator.
    'Cer 18:1;2_16:0' -> 'Cer(d18:1/16:0)'
    'SM  18:1;2_18:0' -> 'SM(d18:1/18:0)'

    WHY '/' IS USED DESPITE '_' INPUT (intentional exception):
    LipidMaps registers ALL sphingolipid molecular subspecies with '/' in d-notation
    (verified: 0 entries with '_', 186+ entries with '/' for Cer alone).
    Biologically, the sphingoid base sn-position in ceramides is fixed by the amide
    bond to the fatty acid — LipidMaps therefore treats these as structural subspecies
    and stores them with '/'.  Using '_' here would yield zero matches.

    This is the ONLY place in the cascade where '_' input generates a '/'-keyed lookup.
    All glycerophospholipids (PC, PE, TG, etc.) have both '_' and '/' entries in
    LipidMaps, so they are correctly matched via to_parentheses_underscore.
    """
    cls, chains, sep = _split_chains(name)
    if cls is None or len(chains) < 2 or cls not in SPHINGOLIPID_CLASSES:
        return None
    first = re.sub(r';.*', '', chains[0])  # strip ';2' or ';O2'
    return f"{cls}(d{first}/{'/'.join(chains[1:])})"


def to_species_v2(name):
    """
    Downgrade to species level (total C:DB), preserving sphingolipid ;O2 modifier.
    'PC 18:1_18:2'    -> 'PC 36:3'
    'Cer 18:1;2_16:0' -> 'Cer 32:1;O2'
    'SM 36:1'         -> 'SM 36:1;O2'  (already species level, but needs ;O2)
    Why: many LipidMaps entries exist only at species level.
    """
    if not re.search(r'\d+:\d+', name):
        return name  # no chain info (e.g. 'Cholesterol') - keep as-is

    cls, chains, sep = _split_chains(name)
    if cls is None:
        return name

    # Sphingolipid already at species level: ensure ;O2 suffix is present
    if cls in SPHINGOLIPID_CLASSES and len(chains) == 1:
        chain = chains[0]
        if ';O' not in chain and ';' not in chain:
            return f"{cls} {chain};O2"
        return name

    if len(chains) == 1:
        return name  # non-sphingolipid already at species level

    pairs    = re.findall(r'(\d+):(\d+)', ' '.join(chains))
    total_c  = sum(int(c)  for c, db in pairs)
    total_db = sum(int(db) for c, db in pairs)

    oh_count = sum(int(m.group(1))
                   for ch in chains
                   for m in [re.search(r';O?(\d+)', ch)] if m)

    species = f"{cls} {total_c}:{total_db}"
    if cls in SPHINGOLIPID_CLASSES:
        n_oh    = oh_count if oh_count >= 2 else 2
        species += f";O{n_oh}"

    return species


def cascade_lookup(name):
    """
    Try each name-format conversion in priority order.
    Return the first hit found in lipid_db, or None if nothing matches.

    Note: for most lipid classes, '_' input stays '_' in the DB key.
    Exception: sphingolipids (Cer, SM, HexCer, ...) always use '/' in LipidMaps
    d-notation — see to_cer_d_notation for the biological justification.
    """
    strategies = [
        ('exact',       lambda n: n),
        ('paren_slash', to_parentheses_slash),
        ('paren_under', to_parentheses_underscore),
        ('cer_d',       to_cer_d_notation),
        ('species',     to_species_v2),
    ]
    for label, fn in strategies:
        converted = fn(name)
        if converted and converted in lipid_db:
            entries = lipid_db[converted]
            return {
                'lm_ids':         [e['lm_id']  for e in entries],
                'smiles_list':    [e['smiles'] for e in entries],
                'converted_name': converted,
                'strategy':       label,
            }
    return None


match_results = [cascade_lookup(n) for n in lipid_names_norm]


## Step 4 — Match rate

In [65]:
n_total   = len(lipid_names)
n_matched = sum(1 for r in match_results if r is not None)
n_failed  = n_total - n_matched

strategy_counts = Counter(
    r['strategy'] for r in match_results if r is not None
)

print(f'Total lipid names  : {n_total}')
print(f'Matched            : {n_matched}  ({n_matched / n_total * 100:.1f}%)')
print(f'Unmatched          : {n_failed}   ({n_failed  / n_total * 100:.1f}%)')
print()
print('Match breakdown by strategy:')
for strat, cnt in strategy_counts.most_common():
    print(f'  {strat:<18} {cnt:>5}  ({cnt / n_total * 100:.1f}%)')

unmatched = [name for name, r in zip(lipid_names, match_results) if r is None]
print(f'\nUnmatched names ({len(unmatched)}):')
for name in unmatched:
    print(f'  {name}')

Total lipid names  : 1289
Matched            : 1259  (97.7%)
Unmatched          : 30   (2.3%)

Match breakdown by strategy:
  paren_under         1062  (82.4%)
  species              110  (8.5%)
  exact                 87  (6.7%)

Unmatched names (30):
  Cer d18:1_18:3
  Cer d18:1_18:4
  Cer d18:1_20:3
  Cer d18:1_20:4
  Cer d18:1_20:5
  Cer d18:1_22:2
  Cer d18:1_22:3
  Cer d18:1_22:4
  Cer d18:1_22:5
  Cer d18:1_22:6
  LPC 22:3
  MG 14:0
  MG 16:1
  MG 20:1
  MG 20:2
  MG 22:0
  PE 22:5_22:6
  PG 22:4_22:5
  PG 22:5_22:6
  HexCer d18:1_22:2
  HexCer d18:1_22:3
  SM 36:4
  SM 36:5
  SM 38:4
  SM 38:5
  SM 38:6
  SM 40:4
  SM 40:5
  SM 40:6
  SM 40:7


In [66]:
# LipidMaps level distribution derived from cascade strategy
# Strategy -> equivalent structural level (LIPID MAPS Shorthand Nomenclature)
STRATEGY_TO_LEVEL = {
    'paren_slash': 'Structural subspecies',   # PC(16:0/18:1) — sn-position known
    'cer_d':       'Structural subspecies',   # Cer(d18:1/14:0) — ceramide structural
    'paren_under': 'Molecular subspecies',    # PC(16:0_18:1)  — sn-position unknown
    'species':     'Species',                 # PC 36:2        — total C:DB only
    'exact':       'Exact match',
}
LEVEL_ORDER_LM = [
    'Structural subspecies',
    'Molecular subspecies',
    'Species',
    'Exact match',
]

level_per_name = {
    name: STRATEGY_TO_LEVEL.get(r['strategy'], r['strategy'])
    for name, r in zip(lipid_names, match_results)
    if r is not None
}

n_total = len(lipid_names)
level_counts = Counter(level_per_name.values())

rows = []
for lv in LEVEL_ORDER_LM:
    n = level_counts.get(lv, 0)
    if n > 0:
        rows.append({'level': lv, 'n': n, 'pct': f'{n / n_total * 100:.1f}%'})

n_unmatched = n_total - len(level_per_name)
rows.append({'level': '(unmatched)', 'n': n_unmatched, 'pct': f'{n_unmatched / n_total * 100:.1f}%'})

df_level_summary = pd.DataFrame(rows)
print(f'LipidMaps level summary (n={n_total}):')
print()
print(df_level_summary.to_string(index=False))

LipidMaps level summary (n=1289):

               level    n   pct
Molecular subspecies 1062 82.4%
             Species  110  8.5%
         Exact match   87  6.7%
         (unmatched)   30  2.3%


In [67]:
# Build result DataFrame and save
rows = []
for orig, norm, result in zip(lipid_names, lipid_names_norm, match_results):
    if result:
        rows.append({
            'original_name':   orig,
            'normalized_name': norm,
            'converted_name':  result['converted_name'],
            'strategy':        result['strategy'],
            'lm_ids':          ';'.join(result['lm_ids']),
            'n_lm_ids':        len(result['lm_ids']),
        })
    else:
        rows.append({
            'original_name':   orig,
            'normalized_name': norm,
            'converted_name':  None,
            'strategy':        None,
            'lm_ids':          None,
            'n_lm_ids':        0,
        })

df_result = pd.DataFrame(rows)

out_path = '../data/processed/lipidmaps_Morita.csv'
df_result.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {df_result.shape}')
df_result.head(10)

Saved: ../data/processed/lipidmaps_Morita.csv
Shape: (1289, 6)


,original_name,normalized_name,converted_name,strategy,lm_ids,n_lm_ids
0,Cer d18:1_14:0,Cer 18:1;2_14:0,Cer 32:1;O2,species,LMSP02010001;LMSP02010034,2
1,Cer d18:1_16:0,Cer 18:1;2_16:0,Cer 34:1;O2,species,LMSP02010004;LMSP02010036;LMSP02010046,3
2,Cer d18:1_16:1,Cer 18:1;2_16:1,Cer 34:2;O2,species,LMSP02010024;LMSP02010037;LMSP02010054;LMSP020...,4
3,Cer d18:1_18:0,Cer 18:1;2_18:0,Cer 36:1;O2,species,LMSP02010006;LMSP02010038;LMSP02010047;LMSP020...,4
4,Cer d18:1_18:1,Cer 18:1;2_18:1,Cer 36:2;O2,species,LMSP02010003;LMSP02010039;LMSP02010048;LMSP020...,6
5,Cer d18:1_18:2,Cer 18:1;2_18:2,Cer 36:3;O2,species,LMSP02010025;LMSP02010057;LMSP02010064,3
6,Cer d18:1_18:3,Cer 18:1;2_18:3,NaN,NaN,NaN,0
7,Cer d18:1_18:4,Cer 18:1;2_18:4,NaN,NaN,NaN,0
8,Cer d18:1_20:0,Cer 18:1;2_20:0,Cer 38:1;O2,species,LMSP02010007;LMSP02010016;LMSP02010040,3
9,Cer d18:1_20:1,Cer 18:1;2_20:1,Cer 38:2;O2,species,LMSP02010026;LMSP02010041;LMSP02010049;LMSP020...,5
